In [ ]:
# ============================================================
# 🧩 STEP 1: Environment Setup
# ============================================================
!pip install -q transformers accelerate datasets evaluate rouge-score bert-score bitsandbytes torch torchvision torchaudio

from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_Token"), add_to_git_credential=True)

from huggingface_hub import login
login(new_session=False)


import os
import gc
import re
import time
import torch
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import evaluate

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# ============================================================
# 🧩 STEP 2: Load full MathQA dataset
# ============================================================
ds = load_dataset("mnlp-nsoai/allenai-math_qa-1000")
print(ds)  # check available splits

# Use the only available split
test_data = ds["train"]
print(f"Number of samples in dataset: {len(test_data)}")
print("Columns:", test_data.column_names)

# ============================================================
# 🧩 STEP 3: Helper Functions
# ============================================================
def extract_option_answer(text):
    """Extract single-letter option a/b/c/d/e from model output"""
    match = re.search(r"\b([a-eA-E])\b", text)
    if match:
        return match.group(1).lower()
    return text.strip().lower()

def calculate_accuracy(preds, refs):
    """Compute accuracy for option selection"""
    correct = sum(p == r for p, r in zip(preds, refs))
    return correct / len(preds)

def compute_metrics(preds, refs):
    """Compute ROUGE, BERTScore, and Exact Match"""
    results = {}

    # ROUGE
    rouge = evaluate.load("rouge")
    rouge_scores = rouge.compute(predictions=preds, references=refs)
    results.update({f"ROUGE_{k}": v for k, v in rouge_scores.items()})

    # BERTScore
    bertscore = evaluate.load("bertscore")
    bert_scores = bertscore.compute(predictions=preds, references=refs, lang="en")
    results["BERTScore_F1"] = sum(bert_scores["f1"]) / len(bert_scores["f1"])

    # Exact match / Accuracy
    results["Exact_Match"] = calculate_accuracy(preds, refs)
    return results

# ============================================================
# 🧩 STEP 4: Batched inference for LLaMA
# ============================================================
def run_inference_batched(model_name, dataset, batch_size=8, quantize_4bit=False):
    torch.cuda.empty_cache()
    gc.collect()

    print(f"\n🚀 Running inference for: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model_kwargs = {
        "device_map": "auto",
        "torch_dtype": torch.float16,
        "low_cpu_mem_usage": True
    }
    if quantize_4bit:
        model_kwargs["load_in_4bit"] = True

    model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
    model.eval()

    # construct prompts
    prompt_prefix = "Choose the correct answer (a/b/c/d/e) without explanation.\nQuestion: "
    inputs = [
        f"{prompt_prefix}{item['question']}\nOptions: {', '.join(item['options'])}\nAnswer:"
        for item in dataset
    ]
    refs = [item["answer"].lower() for item in dataset]
    preds = []

    start_time = time.time()
    for i in tqdm(range(0, len(inputs), batch_size), desc="Generating answers (batched)"):
        batch_inputs = inputs[i:i+batch_size]
        tokenized = tokenizer(batch_inputs, return_tensors="pt", padding=True, truncation=True).to(device)
        with torch.no_grad():
            output = model.generate(**tokenized, max_new_tokens=32)
        for out in output:
            pred = tokenizer.decode(out, skip_special_tokens=True)
            preds.append(extract_option_answer(pred))

    inference_time = time.time() - start_time
    results = compute_metrics(preds, refs)
    results["Accuracy"] = calculate_accuracy(preds, refs)
    results["Inference_Time(s)"] = round(inference_time, 2)
    gpu_mem = torch.cuda.max_memory_allocated(device) / (1024**3)
    model_size_gb = sum(p.numel() for p in model.parameters()) * 2 / (1024**3)
    results["Model_Size(GB)"] = round(model_size_gb, 2)
    results["Peak_GPU_Memory(GB)"] = round(gpu_mem, 2)

    df_model = pd.DataFrame([results])
    file_name = f"{model_name.replace('/', '_')}_mathqa_results_full.csv"
    df_model.to_csv(file_name, index=False)
    print(f"✅ Saved results for {model_name} to {file_name}")

    del model, tokenized, output
    torch.cuda.empty_cache()
    gc.collect()
    return results

# ============================================================
# 🧩 STEP 5: Run LLaMA models on full dataset
# ============================================================
llama_models = [
    ("meta-llama/Llama-3.2-1B-Instruct", False),  # 1B
    ("meta-llama/Llama-2-7b-hf", True)           # 7B with 4-bit
]

llama_results = {}
for model_name, quantize in llama_models:
    batch_size = 8 if "1B" in model_name else 1
    llama_results[model_name] = run_inference_batched(
        model_name, test_data,
        batch_size=batch_size, quantize_4bit=quantize
    )

# ============================================================
# 🧩 STEP 6: Display Results
# ============================================================
df_llama = pd.DataFrame(llama_results).T
print("\n📊 Baseline Vanilla Inference Results (full MathQA dataset) for LLaMA models:")
display(df_llama)


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 18.8 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Device: cuda


README.md:   0%|          | 0.00/423 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/188k [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

Generating answers (batched): 100%|██████████| 800/800 [43:30<00:00,  3.26s/it]
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Saved results for meta-llama/Llama-2-7b-hf to meta-llama_Llama-2-7b-hf_mathqa_results_full.csv

📊 Baseline Vanilla Inference Results (full MathQA dataset) for LLaMA models:


,ROUGE_rouge1,ROUGE_rouge2,ROUGE_rougeL,ROUGE_rougeLsum,BERTScore_F1,Exact_Match,Accuracy,Inference_Time(s),Model_Size(GB),Peak_GPU_Memory(GB)
meta-llama/Llama-3.2-1B-Instruct,0.2025,0.0,0.20125,0.20125,0.964249,0.20125,0.20125,100.67,2.30,3.31
meta-llama/Llama-2-7b-hf,0.2025,0.0,0.20125,0.20125,0.964249,0.20125,0.20125,2610.17,6.52,5.90
